<a href="https://colab.research.google.com/github/katyyass000/Ozymok_4_9_lab/blob/main/%D0%9E%D0%B7%D0%B8%D0%BC%D0%BE%D0%BA_4_9_%D0%9B%D0%A09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** ___Озимок Катерина Сергіївна_______________________  
**Група:** _____________4-9____________________  
**Дата:** ____24.11______________________________

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [3]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [4]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [5]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [6]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
n_users = ratings_df['user_id'].nunique()
n_items = ratings_df['item_id'].nunique()
n_ratings = len(ratings_df)
sparsity = 1 - n_ratings / (n_users * n_items)

print(f"Унікальних користувачів: {n_users}")
print(f"Унікальних фільмів:      {n_items}")
print(f"Усього оцінок:           {n_ratings}")
print(f"Розрідженість матриці:   {sparsity:.2%}\n")

print("Статистика рейтингу:")
print(f"  Мін:     {ratings_df['rating'].min()}")
print(f"  Макс:    {ratings_df['rating'].max()}")
print(f"  Середнє: {ratings_df['rating'].mean():.4f}")
print(f"  Медіана: {ratings_df['rating'].median()}\n")

print("Розподіл оцінок:")
print(ratings_df['rating'].value_counts().sort_index(), "\n")

print("Пропущені значення по колонках:")
print(ratings_df.isna().sum())

Унікальних користувачів: 943
Унікальних фільмів:      1682
Усього оцінок:           100000
Розрідженість матриці:   93.70%

Статистика рейтингу:
  Мін:     1.0
  Макс:    5.0
  Середнє: 3.5299
  Медіана: 4.0

Розподіл оцінок:
rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64 

Пропущені значення по колонках:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [7]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print(f"train_df: {train_df.shape}")
print(f"test_df:  {test_df.shape}")

train_df: (80000, 4)
test_df:  (20000, 4)


In [8]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = list(
    test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None)
)

print(f"Кількість записів у train_data: {len(train_data.raw_ratings)}")
print(f"Кількість записів у testset:    {len(testset)}")
print(f"Приклад елемента testset:       {testset[0]}")

Кількість записів у train_data: 80000
Кількість записів у testset:    20000
Приклад елемента testset:       ('877', '381', 4.0)


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [10]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

Для SVD залишено базові значення параметрів: n_factors (50 і 100) визначає кількість прихованих факторів для опису вподобань, lr_all (0.005 і 0.01) задає швидкість навчання, а reg_all (0.02 і 0.05) зменшує ризик перенавчання моделі.

Для KNNBasic перебираємо кількість сусідів k (20 і 40) та мінімальну їх кількість min_k (1 і 5), щоб уникнути неточних прогнозів на основі поодиноких збігів. Також порівнюємо косинусну схожість і кореляцію Пірсона та обидва варіанти пошуку — за користувачами (user_based=True) і за фільмами (user_based=False), щоб визначити, що краще спрацює на цьому наборі даних.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [11]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_svd.fit(train_data)


In [14]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=1,
    joblib_verbose=1
)
grid_search_knn.fit(train_data)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the pearson similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Comput

[Parallel(n_jobs=1)]: Done  48 out of  48 | elapsed:  3.3min finished


In [15]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
def get_best_cv_summary(gs, model_name):
    best_idx = gs.best_index['rmse']
    return {
        'Model': model_name,
        'Best CV RMSE': round(gs.best_score['rmse'], 4),
        'CV MAE (at best RMSE)': round(gs.cv_results['mean_test_mae'][best_idx], 4),
        'Best CV MAE': round(gs.best_score['mae'], 4),
        'Best Params (RMSE)': gs.best_params['rmse']
    }

cv_summary = pd.DataFrame([
    get_best_cv_summary(grid_search_svd, 'SVD'),
    get_best_cv_summary(grid_search_knn, 'KNNBasic')
])

display(cv_summary)

,Model,Best CV RMSE,CV MAE (at best RMSE),Best CV MAE,Best Params (RMSE)
0,SVD,0.9436,0.7445,0.7445,"{'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0..."
1,KNNBasic,1.0291,0.8161,0.8161,"{'k': 40, 'min_k': 1, 'sim_options': {'name': ..."


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | ... | ... | ... |
| KNNBasic | ... | ... | ... |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [16]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код: # TODO 6.1 — Найкращі гіперпараметри за CV RMSE
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

# TODO 6.2 — Повний навчальний набір
trainset = train_data.build_full_trainset()

# TODO 6.3 — Ініціалізація та навчання остаточних моделей
best_svd = SVD(**best_svd_params, random_state=42)
best_svd.fit(trainset)

best_knn = KNNBasic(**best_knn_params)
best_knn.fit(trainset)

print("Обидві моделі успішно навчено на повній навчальній вибірці.")

Computing the cosine similarity matrix...
Done computing similarity matrix.
Обидві моделі успішно навчено на повній навчальній вибірці.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [18]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код: # TODO 7.1 — Передбачення на тестовій вибірці
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

# TODO 7.2 — Обчислення RMSE та MAE
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

In [19]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame([
    {'Model': 'SVD', 'RMSE': svd_rmse, 'MAE': svd_mae},
    {'Model': 'KNNBasic', 'RMSE': knn_rmse, 'MAE': knn_mae}
]).sort_values(by='RMSE', ascending=True).reset_index(drop=True)

display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | ... | ... |
| KNNBasic | ... | ... |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

На незалежній тестовій вибірці метрики зазвичай виявляються трохи кращими (нижчими) за CV-оцінки, оскільки остаточні моделі навчалися на 100% навчальної вибірки (80 000 оцінок), а не на двох третинах (~53 333 оцінки) під час 3-кратної кросвалідації. Близькість значень CV і Test свідчить про відсутність сильного перенавчання.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [20]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код: user_id = '196'

# TODO 8.1 — Вибір найкращої моделі за CV RMSE
if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']:
    best_model_name = 'SVD'
    best_model = best_svd
else:
    best_model_name = 'KNNBasic'
    best_model = best_knn

print(f"Обрана модель за CV RMSE: {best_model_name}")

# TODO 8.2 — Перевірка наявності користувача у trainset
try:
    inner_uid = trainset.to_inner_uid(user_id)
    user_exists = True
except ValueError:
    user_exists = False
    print(f"Увага: користувач '{user_id}' відсутній у навчальній вибірці (проблема холодного старту).")

if user_exists:
    # TODO 8.3 — Пошук фільмів, які користувач ще не оцінював у trainset
    rated_inner_iids = {iid for (iid, _) in trainset.ur[inner_uid]}
    unrated_raw_iids = [
        trainset.to_raw_iid(iid)
        for iid in trainset.all_items()
        if iid not in rated_inner_iids
    ]

    # TODO 8.4 — Прогнозування та формування топ-10 рекомендацій
    predictions_for_user = [
        (iid, best_model.predict(uid=user_id, iid=iid).est)
        for iid in unrated_raw_iids
    ]

    top_10_df = (
        pd.DataFrame(predictions_for_user, columns=['item_id', 'predicted_rating'])
        .sort_values(by='predicted_rating', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

    print(f"\nТоп-10 рекомендацій для користувача {user_id} (модель {best_model_name}):")
    display(top_10_df)
else:
    popular_fallback = (
        train_df.groupby('item_id')['rating']
        .agg(['count', 'mean'])
        .query('count >= 50')
        .sort_values(by='mean', ascending=False)
        .head(10)
        .reset_index()
        .rename(columns={'mean': 'predicted_rating'})[['item_id', 'predicted_rating']]
    )
    print("\nРекомендації за замовчуванням (популярні фільми з високим рейтингом):")
    display(popular_fallback)

Обрана модель за CV RMSE: SVD

Топ-10 рекомендацій для користувача 196 (модель SVD):


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [ ] Дані `ml-100k` завантажено та перевірено.
- [ ] Є розбиття 80/20 з фіксованим `random_state`.
- [ ] `Reader`, `train_data` і `testset` створені коректно.
- [ ] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [ ] Для SVD і KNNBasic показано параметри та результати CV.
- [ ] Моделі навчено на повній навчальній частині.
- [ ] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [ ] Наведено порівняльну таблицю та інтерпретацію.
- [ ] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [ ] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**